In [ ]:
!pip install torch torchvision transformers accelerate bitsandbytes pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 35.2 MB/s eta 0:00:00


In [ ]:
from custom_llava_new import PrunableLlavaForConditionalGeneration
from pruner.pruner import QueryAwarePruner

ModuleNotFoundError: No module named 'custom_llava_new'

In [ ]:
from transformers import BitsAndBytesConfig, AutoProcessor
import torch

# ----------------------------
# 2) Load model + processor
# ----------------------------
model_id = "llava-hf/llava-1.5-7b-hf"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(model_id)

model = PrunableLlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
)

# Initialize a pruner and attach it to the model
my_pruner = QueryAwarePruner(dim=4096).to(device=model.device, dtype=torch.float16)
model.pruner = my_pruner

model.eval()

In [ ]:
from PIL import Image

# ----------------------------
# 3) Prepare one example
# ----------------------------
image = Image.open("dog.jpg").convert("RGB")

In [ ]:
prompt = "USER: <image>\nWhat is in this image?\nASSISTANT:"

inputs = processor(
    images=image,
    text=prompt,
    return_tensors="pt"
)

# move tensors to model device
for k, v in inputs.items():
    if hasattr(v, "to"):
        inputs[k] = v.to(model.device)


In [ ]:
# ----------------------------
# 4) Run ONE forward pass first
# ----------------------------
with torch.no_grad():
    out = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True,
        use_cache=True,
    )

print("Original image token count:", out.original_image_token_count)
print("Pruned image token count:  ", out.pruned_image_token_count)
print("Pruned image_hidden_states:", out.image_hidden_states.shape)
print("Logits shape:", out.logits.shape)

---

In [ ]:
import torch
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from PIL import Image

model_id = "llava-hf/llava-1.5-7b-hf"

# 4-bit config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

processor = AutoProcessor.from_pretrained(model_id)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)


In [ ]:
image = Image.open("dog.jpg").convert("RGB")

In [ ]:
prompt = "USER: <image>\nWhat is in this image?\nASSISTANT:"

inputs = processor(
    text=prompt,
    images=image,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50)

print(processor.batch_decode(output, skip_special_tokens=True)[0])